In [4]:
%cd /Users/amarmesic/Documents/tudelft/thesis/DNANet

/Users/amarmesic/Documents/tudelft/thesis/DNANet


/Users/amarmesic/miniconda3/envs/dnanet/lib/python3.10/site-packages/IPython/core/magics/osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


In [2]:
batch_used = 1

In [ ]:
import train
train.run(
    "dnanet_rd.yaml",
    "unet_advanced.yaml",
    "training_config.yaml",
    split=batch_used,
    validation_config=0.1,
    checkpoint_dir="output/ProvedIt_best_unet/20250926_142826/checkpoint",
    output_dir="output/result"
    seed=0
)

2025-10-03 10:58:08 INFO     Logs will be written to output/result/log_training.txt
2025-10-03 10:58:08 INFO     Loading model...
2025-10-03 10:58:08 INFO     Loading previous model checkpoint from output/ProvedIt_best_unet/20250926_142826/checkpoint
2025-10-03 10:58:08 INFO     Loading dataset...
2025-10-03 10:58:08 INFO     Loading data from file
2025-10-03 10:58:08 INFO     Walking directory to retrieve all hid files...
Processing folders: 100%|██████████| 20/20 [00:00<00:00, 2576.99it/s]
2025-10-03 10:58:08 INFO     Found 350 HIDs
2025-10-03 10:58:08 INFO     Found 0 HIDs without ladder
2025-10-03 10:58:08 INFO     Removed 0 HIDs without annotation
Loading data from resources/data/2p_5p_Dataset_NFI/Raw data .HID files:  90%|█████████ | 315/350 [00:19<00:02, 13.44it/s]2025-10-03 10:58:27 WARNING  Skipping image: Missing data (resources/data/2p_5p_Dataset_NFI/Raw data .HID files/Mixture dataset 5/Inj7 2017-05-12-16-25-53-867/5E4_C08_08.hid)
Skipping image: Missing data (resources/dat

In [5]:
import evaluate

# results = evaluate.run(
#     data_config="dnanet_rd.yaml",
#     model_config="unet_advanced.yaml",
#     evaluation_config="segmentation.yaml",
#     checkpoint_dir="output/result",
#     split=batch_used,
#     seed=0,
#     save_preds=True
# )

In [ ]:
import train
import evaluate
import neptune
import numpy as np

all_results = []

for batch_used in range(1, 7):  # 1 to 6 inclusive
    for seed in range(3):       # 0, 1, 2
        # Train
        train.run(
            "dnanet_rd.yaml",
            "unet_advanced.yaml",
            "training_config.yaml",
            split=batch_used,
            validation_config=0.1,
            checkpoint_dir="output/ProvedIt_best_unet/20250926_142826/checkpoint",
            output_dir=f"output/result-batch-{batch_used}-seed-{seed}",
            seed=seed
        )
        # Evaluate
        results = evaluate.run(
            data_config="dnanet_rd.yaml",
            model_config="unet_advanced.yaml",
            evaluation_config="segmentation.yaml",
            checkpoint_dir=f"output/result-batch-{batch_used}-seed-{seed}",
            split=batch_used,
            seed=seed,
        )
        all_results.append(results)

        # Neptune logging
        run = neptune.init_run(
            name=f"DNANet的长-1正真:0正假-考试倍{batch_used}-预：训练,缩放-种{seed}",
            project="amar-mesic/dna-thesis",
            api_token="eyJhcGlfYWRkcmVzcyI6Imh0dHBzOi8vYXBwLm5lcHR1bmUuYWkiLCJhcGlfdXJsIjoiaHR0cHM6Ly9hcHAubmVwdHVuZS5haSIsImFwaV9rZXkiOiJkOTQ1Njc4MC0yOTcyLTRlMmQtYTMwMy0xOGYxZTAwMmIzZGUifQ==",
        )
        run["test/pixel_f1"] = float(f"{results['pixel_f1_score']:.4g}")
        run["test/pixel_precision"] = float(f"{results['pixel_precision']:.4g}")
        run["test/pixel_recall"] = float(f"{results['pixel_recall']:.4g}")
        run["test/allele_f1"] = float(f"{results['allele_f1_score']:.4g}")
        run["test/allele_precision"] = float(f"{results['allele_precision']:.4g}")
        run["test/allele_recall"] = float(f"{results['allele_recall']:.4g}")
        meta = {
            "experiment": "Cross-kit-FineTune",
            "model": "Amar-DNANet_Advanced",
            "dataset": "PT:Synth+ProvedIt-FT:NFI_R&D",
            "fold": batch_used,
            "seed": seed,
        }
        for k, v in meta.items():
            run[f'meta/{k}'] = v
        run.stop()

# Compute mean and std for each metric
metrics = ["pixel_f1_score", "pixel_precision", "pixel_recall", "allele_f1_score", "allele_precision", "allele_recall"]
summary = {}
for metric in metrics:
    values = [r[metric] for r in all_results]
    summary[metric] = {
        "mean": np.mean(values),
        "std": np.std(values)
    }

print(summary)

2025-10-03 12:02:51 INFO     Logs will be written to output/result-batch-1-seed-0/log_training.txt
2025-10-03 12:02:51 INFO     Loading model...
2025-10-03 12:02:51 INFO     Loading previous model checkpoint from output/ProvedIt_best_unet/20250926_142826/checkpoint
2025-10-03 12:02:51 INFO     Loading dataset...
2025-10-03 12:02:51 INFO     Loading data from file
2025-10-03 12:02:51 INFO     Walking directory to retrieve all hid files...
Processing folders: 100%|██████████| 20/20 [00:00<00:00, 4809.98it/s]
2025-10-03 12:02:51 INFO     Found 350 HIDs
2025-10-03 12:02:51 INFO     Found 0 HIDs without ladder
2025-10-03 12:02:51 INFO     Removed 0 HIDs without annotation
Loading data from resources/data/2p_5p_Dataset_NFI/Raw data .HID files:  89%|████████▉ | 313/350 [00:17<00:01, 18.51it/s]2025-10-03 12:03:09 WARNING  Skipping image: Missing data (resources/data/2p_5p_Dataset_NFI/Raw data .HID files/Mixture dataset 5/Inj7 2017-05-12-16-25-53-867/5E4_C08_08.hid)
Skipping image: Missing data